# Where is CellOT good on new donors, and why the plateau? (stim maps, held-out donors only)

**Question (2026-10-06).** The learning curve plateaus at a median skill of ~0.6 on held-out donors, against a
split-half ceiling of ~0.87. Is there any stim, celltype or marker where CellOT predicts a *new* donor clearly better
than the plateau, especially on the **patient-specific** part of the response (the shared part can be fixed by a
delta shift)? Where is it weak, and is that biology, noise, or out-of-distribution donors? What would get past the
plateau, or why is it not possible?

Only held-out donors are scored: every map is trained on k donors and scored on 4 others.

## Answer (2026-10-06, updated 2026-10-07 with §7–§8)

**Why the plateau.** At k = 15, the error left by the shift baseline is **95 % real donor-to-donor differences in
the response and only 5 % sampling noise** (§2). So the plateau is not a measurement limit. Each donor responds
differently, and that difference is mostly **not visible in her unstimulated cells**:
- CellOT captures **4.6 %** of it, gauss_ot −4 %, quantile_map −36 %;
- explicit regressions on the donor's Unstim profile (this celltype, or the whole blood) capture 3–8 % (`work/lc/residual_predict`);
- donor contexts (PCs, a fine-tuned encoder, phenotype clusters) add nothing (`work/lc/ctx_score`).

CellOT's donor-specific predictions point the right way (correlation 0.30) but are too weak and too noisy to remove much error.
Going from 3 to 15 training donors moves the specific part only from −3 % to +5 %. This is an **information
limit of the input**, not a model-capacity limit.

**Where CellOT is clearly above the plateau** on the patient-specific part (specific R², positive in most of the 5
repeats):
- **IL-2/4/6 in effector/memory CD8, NKT and CD56hi NK cells:** CD8 Th1 0.51, CD8Tcm CCR2+ 0.47, NKT 0.43,
  CD56hi NK 0.41, CD8Tem CCR2+ 0.39;
- **TNFa in naive/memory T cells:** CD4Tnaive 0.42, CD4Tcm 0.42, CD4Tem 0.37, CD8Tnaive 0.33. These are positive in
  5/5 repeats and tied with gauss_ot;
- **IFNa in CD4Tnaive (0.34) and Granulocytes (0.31); LPS in CD8 Th1 (0.34) and CD4Tnaive (0.31).**

By marker it is the **canonical pathway readout of each stim**:
- pSTAT1 (0.27 overall; IFNa 0.33, LPS 0.31);
- pSTAT6 under GMCSF (0.53) and IFNa (0.34);
- pSTAT3 under TNFa (0.38) and GMCSF (0.30);
- pERK under LPS (0.33).

So where the stim triggers a strong, specific signalling response in a well-defined lymphoid population, the donor's
own response is partly predictable from her Unstim cells.

**Where it is clearly below a plain shift, and why:**
- **PI (−0.09 overall), in monocytes, mDCs, NK cells and CD4 Tregs/Teff.** This is gating, not biology: 23 of the 168
  maps have populations that change size ≥ 1.4× under their stim. Their median specific R² is −0.05, against +0.09
  for stable populations. These include PI cMCs −0.46 (monocytes lost), PI mDCs −0.29 (mDCs 5×) and PI CD4Teff −0.20.
  The "response" there is partly a change of who is in the gate, which the donor's Unstim cells cannot predict.
- **Test donors outside the training range:** where her Unstim value is > 2 training SDs away, CellOT is worse than
  the shift (−0.17 at 2–3 SD, −0.45 beyond 3 SD). It extrapolates with too large an amplitude (SD ratio 1.15). This
  is the out-of-sample failure, and it is what the OOL women face (their inputs sit 2–5 atlas SDs off, see
  `atlas_vs_ool.ipynb`).
- **IL-2/4/6 in CD11c− NK cells, DN T cells, CD56lo NK and B cells (−0.15 to −0.69):** CellOT predicts large donor
  differences (SD ratio 0.86–1.04) with no relation to the truth (correlation 0.04).
- **CD25 in every stim (−0.18; LPS −0.90):** mostly noise (ceiling 0.64), but CellOT predicts a large spread.
- **IL33:** nothing real to predict (ceiling 0.64).
- **Small responses** (mean response < 0.1): negative, while responses > 0.5 reach +0.07. **Pregnant test donors**
  0.03 vs 0.07 for HV. **Few cells**: the bottom cell quartile 0.02 vs 0.09.

**What gets past the plateau now (§5), without a new model:** keep the shift for the average response, add CellOT's
donor-specific prediction only where it held on the other repeats ("gated"), or scaled by a factor learnt on the
other repeats.
- Skill: 0.673 / 0.676, against 0.651 for CellOT and 0.664 for the shift.
- Error: 13 % below CellOT, 7 % below the shift.
- Specific R²: 0.10.
- It beats CellOT for 17 of 20 donor-repeats, but the shift for only 10–12 of 20.

So the gain over a no-network shift is real but small.

**What could move it further.** Each follows from where the signal is and is not:
- **shrink** the donor-specific output, especially for out-of-range donors (amplitude, not direction, is what hurts);
- **don't trust maps on populations that change size** under the stim; handle them at the gate/frequency level;
- **more donors**: slow;
- **an input that carries the donor's responsiveness.** The Unstim cells barely do; a measured response to one stim
  would be the obvious candidate. The cross-celltype correlation notebook found that donor responses are coupled
  across celltypes.

**Two retraining fixes tested (2026-10-07, §7–§8). Neither helps.** Both are paired tests over the 20 test donor-repeats. Skill is on the learning-curve scale, against CellOT 0.596, paired control 0.615 and shift 0.604.
- **Per-donor normalisation (§7).** Each donor's cells are expressed relative to her own Unstim median.
  - It removes the pooled-OT stretch: on stretched features the slope falls from 1.92 to 1.01, against a true 0.69.
  - But the donor-specific signal goes with it. Specific R² falls from +0.045 to −0.14, and the arm is worse than the paired control for 19/20 donor-repeats (skill 0.586; with clipping 0.587, 18/20).
  - Clipping test cells to the training range is harmless. It is better for 16/20 donor-repeats, but by only 0.9 % (skill 0.612). It does not fix out-of-range donors, where the shift stays clearly better.
- **CondOT-style sharing of one map across stims (§8).** The map is conditioned on the perturbation, not the donor.
  - Per celltype, it is worse than per-pair CellOT for 19/20 donor-repeats (skill 0.562).
  - One map for all 168 pairs is worse for 20/20 (skill 0.421, under-fit).
  - The loss is entirely in the average response. The donor-specific part is unchanged.
  - There is no gain for weak or small pairs either (IL33, TNFa, small responses, small celltypes).

So the plateau is not a property of how the per-pair map is set up. The best predictor remains a shift for the average plus a damped CellOT donor-specific part (§5; also the 0.5 CellOT + 0.5 shift ensemble, 0.627, in `mfm_heldout`).

**Caveats.**
- The 14 test donors appear in more than one repeat, so the cross-fitted gating is not fully donor-disjoint.
- Single (stim, celltype, marker) cells rest on 40–100 cases from 14 donors, so the top-20 list is a list of
  candidates.
- The "shared part" of 4 test donors also contains part of their own deviation from the population, so not all of it
  is fixable by a delta.

## 1. Definitions

For a test donor, one (stim, celltype, marker, quantile q):
- *u* = her Unstim quantile, *t* = her stimulated quantile (the truth), response = t − u.
- **shift** baseline = training-mean stimulated value + (u − training-mean Unstim value): every donor gets the
  **average response** of the training donors on top of her own Unstim value. No network, no patient-specific response.
- **Donor-specific response** r = t − shift: how much her response differs from the training average.
- Over the 4 test donors of a repeat, r = **shared part** (their mean: how far these 4 donors' average response is
  from the training average) + **patient-specific part** rc (each donor's deviation from that mean).
- A map's own donor-specific prediction is (prediction − shift), split the same way.
- **specific R²** = 1 − Σ(predicted rc − true rc)² / Σ(true rc)²: the share of the real between-donor differences in
  response that the map captures. 0 = no better than giving every donor the same response; negative = worse.
- **ceiling** = 1 − (sampling noise) / Σ rc², with the noise from split halves of her stim and Unstim cells. It is the
  best reachable specific R², so it says whether the between-donor differences are real or noise.
- **corr** = correlation of predicted and true rc (direction); **SD ratio** = how much the map varies between donors
  relative to the truth (amplitude).
- **skill** (the learning-curve readout) = 1 − |prediction − t| / |t − u|, median per repeat, mean over repeats.
Cases: effect |t − u| > 0.0273 (the technical floor) and at least 3 test donors in the group.

In [ ]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_rows", 120)
WORK = Path("/scratch/users/bmerwan/atlas_heldout/work")
FIG = WORK / "lc/where_skill"
FLOOR = 0.0273
D = pd.read_pickle(WORK / "lc/where_skill/cases.pkl")
D = D[(D.t - D.u).abs() > FLOOR].copy()
GRP = ["repeat", "stim", "celltype", "k", "q", "marker"]
D["n"] = D.groupby(GRP).t.transform("size"); D = D[D.n >= 3].copy()
D["r"] = D.t - D["shift"]; D["rbar"] = D.groupby(GRP).r.transform("mean"); D["rc"] = D.r - D.rbar
D["nv"] = (D.noise ** 2 + D.noise_u ** 2) / 4 * (D.n - 1) / D.n
METH = ["CellOT", "gauss_ot", "quantile_map"]
for m in METH:
    rh = D[m] - D["shift"]
    D["hb_" + m] = rh.groupby([D[c] for c in GRP]).transform("mean"); D["h_" + m] = rh - D["hb_" + m]
D15 = D[D.k == 15].copy()
print(f"{len(D):,} cases over k = 3/5/8/15; at k = 15: {len(D15):,} cases, {D15.donor.nunique()} distinct test donors, "
      f"{D15.groupby(['stim', 'celltype']).ngroups} (stim, celltype) maps")

def spec(g, m="CellOT"):
    sst = (g.rc ** 2).sum()
    return pd.Series({"specific R²": 1 - ((g["h_" + m] - g.rc) ** 2).sum() / sst, "ceiling": 1 - g.nv.sum() / sst,
                      "corr": np.corrcoef(g["h_" + m], g.rc)[0, 1], "SD ratio": np.sqrt((g["h_" + m] ** 2).sum() / sst),
                      "cases": len(g)})

def skill(p, g): return (1 - (p - g.t).abs() / (g.t - g.u).abs()).groupby(g.repeat).median().mean()

def table(g, by, extra=True):
    T = g.groupby(by).apply(spec)
    if extra:
        T["gauss_ot specific R²"] = g.groupby(by).apply(lambda x: spec(x, "gauss_ot")["specific R²"])
        R = g.groupby(by + ["repeat"] if isinstance(by, list) else [by, "repeat"]).apply(lambda x: spec(x)["specific R²"])
        T["repeats > 0 (of 5)"] = (R > 0).groupby(level=list(range(R.index.nlevels - 1))).sum()
    return T

## 2. The plateau: how big is the patient-specific part, and how much of it is noise?

Error budget at k = 15 (mean squared error per case), and the learning curve split into its two parts.

In [ ]:
g = D15
B = pd.DataFrame({
    "shift": {"shared part": (g.rbar ** 2).mean(), "patient-specific part": (g.rc ** 2).mean()},
    "CellOT": {"shared part": ((g.hb_CellOT - g.rbar) ** 2).mean(), "patient-specific part": ((g.h_CellOT - g.rc) ** 2).mean()},
    "gauss_ot": {"shared part": ((g.hb_gauss_ot - g.rbar) ** 2).mean(), "patient-specific part": ((g.h_gauss_ot - g.rc) ** 2).mean()}}).T
B["total"] = B.sum(1)
B.loc["sampling noise (in the truth)"] = [np.nan, g.nv.mean(), g.nv.mean()]
display(B.round(4))
print("overall, k = 15:"); display(pd.DataFrame({m: spec(g, m) for m in METH}).T.round(3))
LC = []
for k, x in D.groupby("k"):
    LC.append({"k": k, "skill shift": skill(x["shift"], x), "skill CellOT": skill(x.CellOT, x),
               "skill gauss_ot": skill(x.gauss_ot, x),
               "rms shared error shift": np.sqrt((x.rbar ** 2).mean()),
               "rms shared error CellOT": np.sqrt(((x.hb_CellOT - x.rbar) ** 2).mean()),
               "specific R² CellOT": spec(x)["specific R²"], "specific R² gauss_ot": spec(x, "gauss_ot")["specific R²"]})
LC = pd.DataFrame(LC).set_index("k"); display(LC.round(3))

**How to read §2.** First table: the mean squared error per case, split into the shared part (how far the 4 test
donors' average is from the prediction's average) and the patient-specific part. The last row is the sampling noise
in the truth, the part nobody can predict. Second: the specific-part statistics at k = 15. Third: the learning
curve, for overall skill and for each part.

**Reading.**
- **The shift's error is 0.0346; noise is 0.0017 of it (5 %).** The rest is real. The patient-specific part alone is
  0.0233, against 0.0017 noise, so the between-donor differences are 93 % real (ceiling 0.93).
- **CellOT's patient-specific error is 0.0222, hardly below the shift's 0.0233** (specific R² 0.046). Its shared part
  is worse than the shift's (0.0150 vs 0.0113), which is why its total error is higher.
- **The direction is right but the amplitude is not.** Correlation 0.30, SD ratio 0.51: CellOT predicts half the true
  between-donor spread, and only part of what it predicts is right.
- **Learning curve:** from k = 3 to 15 the shared error falls (0.138 → 0.106), which is what more donors mainly buy.
  The specific R² moves −0.03 → +0.05, still rising but slowly. gauss_ot stays negative.

## 3. Where CellOT captures the patient-specific response (k = 15)

By stim, quantile, marker and celltype. *repeats > 0* = in how many of the 5 repeats the specific R² is positive.

In [ ]:
for by in ["stim", "q", "marker", "celltype"]:
    display(table(D15, by).sort_values("specific R²", ascending=False).round(3))

In [ ]:
ST = table(D15, ["stim", "celltype"]); SM = table(D15, ["stim", "marker"], extra=False)
fig, axs = plt.subplots(1, 2, figsize=(14, 8.5), gridspec_kw={"width_ratios": [7, 7]})
for ax, T, rows, title in [(axs[0], ST, "celltype", "stim × celltype"), (axs[1], SM, "marker", "stim × marker")]:
    X = T["specific R²"].unstack(0)
    if rows == "celltype": X = X.loc[X.mean(1).sort_values().index]
    else: X = X.loc[X.mean(1).sort_values().index]
    im = ax.imshow(X.values, cmap="RdBu_r", vmin=-0.5, vmax=0.5, aspect="auto")
    ax.set_xticks(range(X.shape[1])); ax.set_xticklabels(X.columns)
    ax.set_yticks(range(X.shape[0])); ax.set_yticklabels(X.index, fontsize=8); ax.set_title(title, fontsize=10)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            v = X.values[i, j]
            if np.isfinite(v) and abs(v) >= 0.25: ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5,
                                                          color="white" if abs(v) > 0.4 else "black")
cb = fig.colorbar(im, ax=axs, shrink=0.6)
cb.set_label("specific R² of CellOT on new donors\n(red: captures real between-donor differences; blue: worse than shift)")
fig.suptitle("Patient-specific part of the stim response, held-out donors, k = 15 (blank = pair not in the grid)", fontsize=10)
fig.savefig(FIG / "specific_heatmaps.png", dpi=160, bbox_inches="tight")

In [ ]:
print("best (stim, celltype) maps"); display(ST.sort_values("specific R²", ascending=False).head(15).round(3))
print("worst (stim, celltype) maps"); display(ST.sort_values("specific R²").head(15).round(3))
SCM = D15.groupby(["stim", "celltype", "marker"]).apply(spec)
SCM = SCM[SCM.cases >= 40]
print("best (stim, celltype, marker), >= 40 cases"); display(SCM.sort_values("specific R²", ascending=False).head(20).round(3))

**How to read §3.** Specific R² by one factor, then the stim × celltype and stim × marker heatmaps (red = CellOT
captures real between-donor differences, blue = worse than the shift; numbers shown where |value| ≥ 0.25), then the
best and worst maps and the best single features. *repeats > 0* counts how many of the 5 repeats agree in sign.

**Reading.**
- **Stims:** IFNa 0.19, IL246 0.14, LPS 0.14 lead; GMCSF 0.08, TNFa 0.03; IL33 −0.03 (nothing to predict, ceiling
  0.64); PI −0.09.
- **Markers:** pSTAT1 0.27 is the best everywhere, then pCREB, pS6, IkB (0.11–0.13). pSTAT5, pSTAT6 and CD25 are
  negative overall, but pSTAT6 is the best marker *within* GMCSF (0.53) and IFNa (0.34). Its negative total comes
  from PI (−0.50).
- **Celltypes:** T and NKT subsets lead (CD8Tcm CCR2+ 0.28, CD8 Th1 0.25, NKT 0.18, CD4Tnaive 0.14). Myeloid and NK
  subsets trail (cMCs −0.18, DN T −0.19, CD11c− NK −0.14).
- **Quantiles:** q25 / q10 / median 0.08–0.11; q90 −0.01. The upper tail is where CellOT's donor-specific output is
  least reliable.
- **Best maps:** IL246 in CD8 Th1 / CD8Tcm CCR2+ / NKT / CD56hi NK / CD8Tem CCR2+ (0.39–0.51), TNFa in naive and
  memory CD4/CD8 T (0.33–0.42, 5/5 repeats), IFNa CD4Tnaive (0.34, 5/5), LPS CD8 Th1 / CD4Tnaive.
  - gauss_ot is as good on the TNFa T-cell maps (0.36–0.45). There the donor-specific signal is affine: CellOT is not
    needed for it.
  - CellOT is clearly ahead of gauss_ot on IL246 CD56hi NK (0.41 vs 0.17), IL246 CD8Tem CCR2+ (0.39 vs 0.14),
    IFNa CD4Tnaive (0.34 vs 0.12) and LPS CD4Tnaive (0.31 vs 0.14).
- **Best single features**, 40–100 cases each, so candidates:
  - **pSTAT6 under GMCSF in every myeloid population (0.59–0.77);**
  - pSTAT6/pSTAT5 under IL246 in NK/NKT/CD8 (0.60–0.75);
  - IkB under TNFa/PI in NK and T cells (~0.57);
  - two PI features (Granulocytes CD25 0.85, CD4Teff HLADR 0.83).

## 4. Why is it weak where it is weak?

Candidate explanations, each checked against the specific R²:
1. **Noise**: the ceiling (is there anything real to predict?) and the number of cells.
2. **Out of distribution**: |z| = how far the test donor's Unstim value is from the training donors, in training SDs.
3. **Gating**: populations whose size changes under the stim (the "response" partly becomes a change of who is in
   the gate). Population size ratio stim / Unstim from `work/heldout/pheno` (plate controls, median over donors).
4. **Cohort**: pregnant (PV) vs non-pregnant (HV) test donors.
5. **Effect size**: how big the average response is.

In [ ]:
x = D15.copy()
x["|z| of test donor Unstim"] = pd.cut(((x.u - x.mu_u) / x.sd_u.replace(0, np.nan)).abs(), [0, 0.5, 1, 2, 3, np.inf])
x["cells (stim well)"] = pd.qcut(x.n_s, 4)
x["cohort"] = np.where(x.donor.str.startswith("PV"), "PV (pregnant)", "HV")
x["mean response"] = pd.cut((x.mu_s - x.mu_u).abs(), [0, 0.05, 0.1, 0.2, 0.5, np.inf])
for by in ["|z| of test donor Unstim", "cells (stim well)", "cohort", "mean response"]:
    T = x.groupby(by).apply(spec)
    T["CellOT closer than shift"] = x.groupby(by).apply(lambda g: ((g.h_CellOT - g.rc).abs() < g.rc.abs()).mean())
    display(T.round(3))

In [ ]:
# population size change under each stim (plate-control wells, same donor and plate)
P = pd.concat([pd.read_csv(f, usecols=["celltype", "stim", "treatment", "donor", "n_cells"]).drop_duplicates()
               for f in sorted((WORK / "heldout/pheno").glob("*.csv.gz"))])
P = P[P.treatment.str.startswith("DMSO")]
P["freq"] = P.n_cells / P.groupby(["stim", "treatment", "donor"]).n_cells.transform("sum")
Pu = P[P.stim == "Unstim"].set_index(["celltype", "treatment", "donor"]).freq
Ps = P[P.stim != "Unstim"].copy()
Ps["l2"] = np.log2(Ps.freq.to_numpy() / Pu.reindex(pd.MultiIndex.from_frame(Ps[["celltype", "treatment", "donor"]])).to_numpy())
SZ = Ps.replace([np.inf, -np.inf], np.nan).groupby(["celltype", "stim", "donor"]).l2.mean().groupby(["stim", "celltype"]).median()
ST2 = ST.join(SZ.rename("log2 size ratio")).dropna(subset=["log2 size ratio"])
ST2["size change"] = pd.cut(ST2["log2 size ratio"].abs(), [-0.01, 0.25, 0.5, np.inf],
                            labels=["< 1.2×", "1.2–1.4×", "≥ 1.4×"])
print("(stim, celltype) maps by how much the population changes size under the stim")
display(ST2.groupby("size change").agg(maps=("cases", "size"), median_specific_R2=("specific R²", "median"),
                                       mean_corr=("corr", "mean")).round(3))
display(ST2[ST2["size change"] == "≥ 1.4×"].sort_values("specific R²")[["specific R²", "corr", "log2 size ratio"]].round(2))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
T = x.groupby("|z| of test donor Unstim").apply(spec)
axs[0].bar(range(len(T)), T["specific R²"], color=["#d62728" if v > 0 else "#1f77b4" for v in T["specific R²"]])
axs[0].set_xticks(range(len(T))); axs[0].set_xticklabels([str(i) for i in T.index], fontsize=8)
axs[0].axhline(0, c="k", lw=.8); axs[0].set_xlabel("|z|: test donor's Unstim value vs training donors (SDs)")
axs[0].set_ylabel("specific R² of CellOT"); axs[0].set_title("Out-of-range donors", fontsize=10); axs[0].grid(axis="y", alpha=.3)
for lab, c in [("< 1.2×", "#2ca02c"), ("1.2–1.4×", "#ff7f0e"), ("≥ 1.4×", "#9467bd")]:
    s = ST2[ST2["size change"] == lab]
    axs[1].scatter(s["log2 size ratio"], s["specific R²"].clip(-1, 1), s=14, c=c, label=f"size change {lab}")
axs[1].axhline(0, c="k", lw=.8); axs[1].set_xlabel("log2 population size, stim / Unstim")
axs[1].set_ylabel("specific R² (clipped at −1)"); axs[1].set_title("One dot per (stim, celltype) map", fontsize=10)
axs[1].legend(fontsize=7, frameon=False); axs[1].grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "why_weak.png", dpi=160)

**How to read §4.** Each table splits the k = 15 cases by one candidate explanation. *CellOT closer than shift* =
share of cases where CellOT's donor-specific error is smaller. The population-size table groups the (stim,
celltype) maps by how much the population grows or shrinks under the stim. Figure, left: by how far the test donor
is from the training donors. Right: one dot per map, x = log2 population size change.

**Reading.**
- **Out of range:** up to 2 training SDs CellOT adds a little (0.04–0.06, closer in 52–54 %). Beyond that it is worse
  than the shift: −0.17 at 2–3 SD, −0.45 beyond 3 SD (closer in only 39 %). Its SD ratio there exceeds 1, so it
  over-extrapolates. This is 6 % of cases here, but OOL inputs are 2–5 atlas SDs off on several phospho-markers.
- **Gating:** maps whose population changes size ≥ 1.4× have a median specific R² of −0.05, against +0.09 for stable
  ones. 16 of the 23 are PI. In the figure every good map (> 0.3) sits at a size change near 0.
- **Noise and cells:** the ceiling is high almost everywhere (0.84–0.99) except few-cell wells (0.84) and small
  responses (0.68–0.73). More cells helps a little (0.02 → 0.09).
- **Response size:** specific R² rises with the mean response (−0.05 for < 0.05, +0.07 for > 0.5). Strong
  stimulations carry more predictable donor variation.
- **Cohort:** HV test donors 0.07, PV 0.03; the direction is similar (correlation 0.28 vs 0.30).

## 5. What gets past the plateau now: shift for the average, CellOT only for the donor-specific part

Three ways to combine the shift with CellOT, all built from training information or from the test donors' *inputs*
(never their truth):
- **hybrid** = shift + CellOT's donor-specific prediction (CellOT's own average response is replaced by the
  shift's, i.e. the delta correction; the centring uses CellOT's predictions on the 4 test donors, not their truth);
- **gated** = hybrid only for the (stim, celltype, marker) where CellOT's specific R² was positive on the *other four*
  repeats, shift elsewhere (cross-fitted);
- **scaled** = shift + β × CellOT's donor-specific prediction, β (clipped to [0, 1.5]) fitted on the other four
  repeats per (stim, celltype, marker).

In [ ]:
g = D15.copy(); g["key"] = list(zip(g.stim, g.celltype, g.marker))
out = []
for r in sorted(g.repeat.unique()):
    tr, te = g[g.repeat != r], g[g.repeat == r].copy()
    st = tr.groupby("key").apply(lambda z: pd.Series({
        "spec_other": 1 - ((z.h_CellOT - z.rc) ** 2).sum() / (z.rc ** 2).sum(),
        "beta": (z.h_CellOT * z.rc).sum() / max((z.h_CellOT ** 2).sum(), 1e-12)}))
    te = te.join(st, on="key"); out.append(te)
O = pd.concat(out)
cands = {"shift": O["shift"], "CellOT": O.CellOT, "gauss_ot": O.gauss_ot, "hybrid": O["shift"] + O.h_CellOT,
         "gated": O["shift"] + np.where(O.spec_other > 0, O.h_CellOT, 0),
         "scaled": O["shift"] + O.beta.clip(0, 1.5).fillna(0) * O.h_CellOT}
sst = (O.rc ** 2).sum(); rows = {}
for n, p in cands.items():
    hc = (p - O["shift"]) - (p - O["shift"]).groupby([O[c] for c in GRP]).transform("mean")
    e = (p - O.t) ** 2
    pdn = e.groupby([O.repeat, O.donor]).mean()
    rows[n] = {"skill": skill(p, O), "specific R²": 1 - ((hc - O.rc) ** 2).sum() / sst, "MSE": e.mean(),
               "MSE vs CellOT": e.mean() / ((O.CellOT - O.t) ** 2).mean() - 1}
    rows[n]["_pdn"] = pdn
R5 = pd.DataFrame({n: {k: v for k, v in d.items() if k != "_pdn"} for n, d in rows.items()}).T
base = rows["shift"]["_pdn"]
R5["donor-repeats better than shift"] = [f"{(rows[n]['_pdn'] < base).sum()}/{len(base)}" for n in rows]
cot = rows["CellOT"]["_pdn"]
R5["donor-repeats better than CellOT"] = [f"{(rows[n]['_pdn'] < cot).sum()}/{len(cot)}" for n in rows]
display(R5.round(4))
print(f"gated: CellOT's donor-specific part is used for {(O.spec_other > 0).mean():.0%} of cases")
print("specific R² of the gated combination by stim:")
pg = cands["gated"]; O["h_gated"] = (pg - O["shift"]) - (pg - O["shift"]).groupby([O[c] for c in GRP]).transform("mean")
display(O.groupby("stim").apply(lambda z: 1 - ((z.h_gated - z.rc) ** 2).sum() / (z.rc ** 2).sum()).round(3).to_frame("gated specific R²"))

**How to read §5.** One row per way of predicting:
- *skill*: as in the learning curve;
- *specific R²*;
- *MSE*, and *MSE vs CellOT* (negative = less error than CellOT);
- *donor-repeats better than …*: in how many of the 20 test donor-repeats the mean squared error is smaller.

The gated and scaled versions are cross-fitted: the choice for repeat r uses only the other four repeats.

**Reading.**
- **Replacing CellOT's average with the shift's ("hybrid") is the cheapest fix.** Error falls 10 % below CellOT,
  skill rises 0.651 → 0.671, and it beats CellOT in 17 of 20 donor-repeats. CellOT's weak point is the average
  response, not the donor-specific one.
- **Using CellOT's donor-specific part only where it held elsewhere ("gated"), or scaled ("scaled"), doubles the
  specific R² to 0.10.** Error is 13 % below CellOT and 7 % below the shift; skill 0.673 / 0.676. It is used for 56 %
  of cases.
- **Against the shift the gain is modest:** better in 10–12 of 20 donor-repeats. Per stim it is IFNa 0.19, LPS 0.16,
  IL246 0.13, TNFa 0.09, and about 0 for GMCSF, PI and IL33.
- So with the current inputs and 15 donors, about 10 % of the real donor-specific response is recoverable, and it
  lives in a known set of stims, celltypes and markers.

## 7. Retraining with per-donor normalisation and input clipping (submitted 2026-10-06)

§3–§4 traced the failures to the pooled-OT stretch: a map between pooled donors sends a cell at a given rank of
pooled Unstim to the same rank of pooled stim, so where the stim cloud is wider for reasons unrelated to baseline
(pSTAT5/6, PI, LPS), it turns a donor's baseline offset into an amplified response. Three paired, context-free
CellOT arms (`lc/cond_train.py --ctx none`, k = 15, same training as the paired control `none`) test the fixes:
- **dnorm**: trained on `cache_dnorm` (each donor's cells minus her own Unstim median, `lc/dnorm_cache.py`);
  prediction + her offset;
- **dnorm_clip**: dnorm, and each test cell clamped per marker to the [0.5, 99.5] percentiles of the map's pooled
  training Unstim cells (`--clip-test 0.5`);
- **clip**: clipping only, raw cells.
Scored by `lc/where_skill.py --ks 15 --extra ...` → `work/lc/where_skill/cases_dnorm.pkl` (slurm/dnorm_score.sbatch).

In [ ]:
def compare(F7, arms):
    if not F7.exists():
        print("pending: training jobs not finished (see work/lc/STATUS.md)"); return
    E = pd.read_pickle(F7)
    E = E[(E.t - E.u).abs() > FLOOR].copy()
    ARMS = [m for m in arms if m in E and E[m].notna().any()]
    E = E.dropna(subset=ARMS)
    E["n"] = E.groupby(GRP).t.transform("size"); E = E[E.n >= 3].copy()
    E["r"] = E.t - E["shift"]; E["rbar"] = E.groupby(GRP).r.transform("mean"); E["rc"] = E.r - E.rbar
    E["nv"] = (E.noise ** 2 + E.noise_u ** 2) / 4 * (E.n - 1) / E.n
    for m in ARMS:
        rh = E[m] - E["shift"]; E["hb_" + m] = rh.groupby([E[c] for c in GRP]).transform("mean"); E["h_" + m] = rh - E["hb_" + m]
    G7 = [E[c] for c in ["repeat", "stim", "celltype", "q", "marker"]]
    E["uc"] = E.u - E.u.groupby(G7).transform("mean"); E["tc"] = E.t - E.t.groupby(G7).transform("mean")
    rows = {}
    base = ((E["shift"] - E.t) ** 2).groupby([E.repeat, E.donor]).mean()
    for m in ["shift"] + ARMS:
        e = (E[m] - E.t) ** 2; pdn = e.groupby([E.repeat, E.donor]).mean()
        row = {"skill": skill(E[m], E), "MSE": e.mean(), "donor-repeats better than shift": f"{(pdn < base).sum()}/{len(base)}"}
        if m != "shift":
            row.update(spec(E, m)[["specific R²", "corr", "SD ratio"]].to_dict())
            row["shared MSE"] = ((E["hb_" + m] - E.rbar) ** 2).mean()
            pc = E[m] - E[m].groupby(G7).transform("mean")
            row["slope vs Unstim (true %.2f)" % ((E.tc * E.uc).sum() / (E.uc ** 2).sum())] = (pc * E.uc).sum() / (E.uc ** 2).sum()
        rows[m] = row
    display(pd.DataFrame(rows).T.round(4))
    print("specific R² by stim"); display(pd.DataFrame({m: E.groupby("stim").apply(lambda g: spec(g, m)["specific R²"]) for m in ARMS}).round(3))
    print("specific R² by |z| of the test donor's Unstim")
    E["|z|"] = pd.cut(((E.u - E.mu_u) / E.sd_u.replace(0, np.nan)).abs(), [0, 1, 2, 3, np.inf])
    display(pd.DataFrame({m: E.groupby("|z|").apply(lambda g: spec(g, m)["specific R²"]) for m in ARMS}).round(3))
    for m in ARMS: E["pc_" + m] = E[m] - E[m].groupby(G7).transform("mean")
    return E

def dr_mse(E, m): return ((E[m] - E.t) ** 2).groupby([E.repeat, E.donor]).mean()

def paired_dr(E, arms, ref):
    # paired over test donor-repeats (20): mean squared error per donor-repeat, arm vs ref
    from scipy.stats import wilcoxon
    b = dr_mse(E, ref); out = {}
    for m in arms:
        if m == ref: continue
        a = dr_mse(E, m)
        out[m] = {f"donor-repeats better than {ref}": f"{(a < b).sum()}/{len(b)}",
                  f"median MSE ratio vs {ref}": np.median(a / b), "Wilcoxon p": wilcoxon(a, b).pvalue}
    return pd.DataFrame(out).T

def skill_by(E, by, arms): return pd.DataFrame({m: E.groupby(by).apply(lambda g: skill(g[m], g)) for m in arms})

E7 = compare(WORK / "lc/where_skill/cases_dnorm.pkl", ["CellOT", "paired", "dnorm", "dnorm_clip", "clip", "gauss_ot"])

In [ ]:
# §7b: paired tests over donor-repeats, and the pooled-OT stretch feature by feature
A7 = ["CellOT", "paired", "dnorm", "dnorm_clip", "clip"]
print("paired over the 20 test donor-repeats, vs the paired control (same training, raw cells)")
display(paired_dr(E7, A7 + ["shift"], "paired").round(4))
F = ["stim", "celltype", "q", "marker"]
slope = lambda g, c: (g[c] * g.uc).sum() / (g.uc ** 2).sum()
S7 = E7.groupby(F).apply(lambda g: pd.Series({"true": slope(g, "tc"), **{m: slope(g, "pc_" + m) for m in A7}}))
S7["stretched"] = (S7.paired - S7.true) > 0.5
print(f"{len(S7)} features (stim, celltype, quantile, marker); 'stretched' = paired control's slope vs Unstim exceeds "
      f"the true slope by > 0.5: {S7.stretched.mean():.1%}")
print("share of features whose slope exceeds the true one by > 0.5, per arm:")
display(pd.Series({m: ((S7[m] - S7.true) > 0.5).mean() for m in A7}).round(3).to_frame("share"))
print("median slope vs Unstim (across test donors) in stretched / other features")
display(S7.groupby("stretched")[["true"] + A7].median().round(3))
E7 = E7.join(S7.stretched, on=F)
print("specific R² and skill in stretched / other features")
display(pd.concat({"specific R²": pd.DataFrame({m: E7.groupby("stretched").apply(lambda g: spec(g, m)["specific R²"]) for m in A7}),
                   "skill": skill_by(E7, "stretched", A7 + ["shift"])}).round(3))
print("paired tests restricted to the stretched features"); display(paired_dr(E7[E7.stretched], A7 + ["shift"], "paired").round(4))
print("slope vs Unstim by stim"); display(E7.groupby("stim").apply(lambda g: pd.Series({"true": slope(g, "tc"), **{m: slope(g, "pc_" + m) for m in A7}})).round(3))
print("skill by stim"); display(skill_by(E7, "stim", A7 + ["shift"]).round(3))
print("specific R² by quantile and by marker")
display(pd.DataFrame({m: E7.groupby("q").apply(lambda g: spec(g, m)["specific R²"]) for m in A7}).round(3))
display(pd.DataFrame({m: E7.groupby("marker").apply(lambda g: spec(g, m)["specific R²"]) for m in A7}).round(3))
print("by |z| of the test donor's Unstim: cases, MSE, and paired tests vs the paired control")
display(pd.concat([E7.groupby("|z|").size().rename("cases"),
                   pd.DataFrame({m: E7.groupby("|z|").apply(lambda g: ((g[m] - g.t) ** 2).mean()) for m in A7 + ["shift"]})], axis=1).round(4))
for b, g in E7.groupby("|z|"):
    print(f"|z| in {b}"); display(paired_dr(g, ["dnorm", "dnorm_clip", "clip", "CellOT"], "paired").round(4))

In [ ]:
COL = {"paired": "#2a78d6", "dnorm": "#eb6834", "dnorm_clip": "#e87ba4", "clip": "#1baf7a", "shift": "#6b6b6b",
       "CellOT": "#2a78d6", "condot_ct": "#eda100", "condot_global": "#eb6834"}
fig, axs = plt.subplots(1, 2, figsize=(11.5, 4))
arms = ["paired", "dnorm", "dnorm_clip", "clip"]
X = S7.groupby("stretched")[["true"] + arms].median()
w = 0.17
for i, m in enumerate(["true"] + arms):
    axs[0].bar(np.arange(2) + (i - 2) * w, X[m], w * 0.9, color="#c3c2b7" if m == "true" else COL[m], label="truth" if m == "true" else m)
axs[0].set_xticks([0, 1]); axs[0].set_xticklabels([f"other features ({1 - S7.stretched.mean():.0%})", f"stretched features ({S7.stretched.mean():.0%})"])
axs[0].axhline(1, c="k", lw=.6, ls=":"); axs[0].set_ylabel("slope of prediction vs donor's Unstim value")
axs[0].set_title("dnorm removes the stretch (slope back to ~1)", fontsize=10); axs[0].legend(fontsize=7, frameon=False)
Z = pd.DataFrame({m: E7.groupby("|z|").apply(lambda g: spec(g, m)["specific R²"]) for m in arms})
for i, m in enumerate(arms):
    axs[1].bar(np.arange(len(Z)) + (i - 1.5) * 0.2, Z[m].clip(-0.8), 0.18, color=COL[m], label=m)
axs[1].set_xticks(range(len(Z))); axs[1].set_xticklabels([f"{i.left:g}–{i.right:g}" for i in Z.index])
axs[1].axhline(0, c="k", lw=.8); axs[1].set_xlabel("|z|: test donor's Unstim value vs training donors (SDs)")
axs[1].set_ylabel("specific R²"); axs[1].set_title("...but loses the donor-specific signal", fontsize=10)
axs[1].legend(fontsize=7, frameon=False)
for ax in axs: ax.grid(axis="y", alpha=.3); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(FIG / "dnorm_stretch.png", dpi=160)

**How to read §7.** First table: one row per arm, all on the same cases.
- *skill*, *MSE*, *specific R²*, *corr* and *SD ratio* are as in §2–§5.
- *shared MSE* is the error on the 4 test donors' average response.
- *slope vs Unstim* measures how strongly the prediction follows the donor's own Unstim value across the test donors. The true slope is in the column name.

Then specific R² by stim and by |z|. §7b gives:
- paired tests over the **20 test donor-repeats**, against the paired control (`paired`: same training, raw cells);
- the stretch feature by feature. A feature is *stretched* when the paired control's slope exceeds the true slope by more than 0.5;
- slope and skill by stim, specific R² by quantile and marker, and MSE and paired tests by |z| bin.

**Scale.** Skill here keeps only groups with at least 3 test donors above the floor. That drops 16 % of cases, mostly small effects, so every level is about 0.06 above the learning curve's. On the learning-curve scale:

| arm | skill |
|---|---|
| shift | 0.604 |
| CellOT | 0.596 |
| paired | 0.615 |
| dnorm | 0.586 |
| dnorm_clip | 0.587 |
| clip | 0.612 |

**Reading (2026-10-07).**
- **Per-donor normalisation does remove the stretch.** In the 15 % of features where the paired control is stretched, its median slope is 1.92 against a true 0.69. With dnorm it is 1.01. The share of features with slope > truth + 0.5 falls from 14.5 % to 6.8 %. By stim, dnorm's slope is 0.99–1.01; the paired control's is 1.00–1.05; the truth is 0.90–0.98.
- **But it removes the donor-specific signal with it, and is worse overall.**
  - Specific R² falls from +0.045 to −0.14 (dnorm_clip −0.11), and corr from 0.29 to −0.03.
  - dnorm is worse than the paired control for **19 of 20** donor-repeats (MSE +12 %, p 4e-6). dnorm_clip is worse for 18/20 (+11 %, p 6e-4).
  - Neither beats the shift on any donor-repeat (0/20).
  - Even inside the stretched features it does not help: specific R² −0.12 vs −0.15, skill 0.744 vs 0.768, better on 6/20.
- **Why.** What CellOT gets right about a donor sits in her baseline level, and that is exactly what normalisation removes. After normalisation the map sees only the shape of her cells around her own median, and that shape does not predict her response.
  - Specific R², paired → dnorm: pSTAT1 0.27 → 0.00, pSTAT3 0.03 → −0.41, pSTAT6 −0.12 → −0.41.
  - Low tails (q10 / q25): 0.09 / 0.10 → −0.43 / −0.45.
  - Worst stim: IL246, −0.48.
  - So the stretch and the useful signal come through the same channel, and removing the channel loses more than it gains.
- **Out-of-range donors.** dnorm is worst beyond 3 SD (specific R² −0.70, MSE 0.097 vs paired 0.071).
- **Clipping alone is harmless and helps slightly.**
  - Better than the paired control for **16/20** donor-repeats, but by only 0.9 % (median MSE ratio 0.991, p 0.006).
  - Skill 0.667 vs 0.666; specific R² 0.051 vs 0.045.
  - The gain is mostly for out-of-range donors: 14/20 at 2–3 SD, 12/18 beyond 3 SD.
  - It does not close the gap with the shift there. Beyond 3 SD the MSE is clip 0.069, shift 0.049. At 2–3 SD it is 0.043 vs 0.039. Clamping single cells to the training range does not move a donor whose whole distribution is shifted.
- **Net.** Neither fix removes the out-of-range failure. The robust way to handle the average response and out-of-range donors is still to fall back on the shift (§5 hybrid/gated; also the 0.5 CellOT + 0.5 shift ensemble in `mfm_heldout`). Clipping can be kept as a free safeguard.

## 8. CondOT-faithful: one map shared across stims, conditioned on the perturbation (submitted 2026-10-06)

CondOT (Bunne, Krause, Cuturi, NeurIPS 2022, github.com/bunnech/condot) conditions the map on a description of the
*perturbation* (dose, drug one-hot, mode of action, combination of genetic perturbations), never on the donor. Its
PICNN (vendored verbatim, MIT, `lc/picnn.py`) lets a context network modulate every layer while staying convex in the
cells. Two arms, trained exactly as CondOT / CellOT (pooled unpaired cells of the 15 training donors, one condition per
iteration, 10 g : 1 f, Adam 1e-4 (0.5, 0.9), uniform(0, 0.1) init), checkpoint = best mean val median error on the
training donors' val cells (`lc/condot_train.py`):
- **condot_ct**: one PICNN per celltype, context = one-hot stim (CondOT's `onehot` experiment), 50k iterations;
- **condot_global**: one PICNN for all 168 pairs, context = one-hot stim + one-hot celltype, 150k iterations.
Compared with the per-pair unpaired CellOT of the learning curve (`CellOT`, same pooling) and the paired control.
The question: does sharing one map across stims (and celltypes) borrow strength, especially for weak or small pairs?

In [ ]:
E8 = compare(WORK / "lc/where_skill/cases_condot.pkl", ["CellOT", "paired", "condot_ct", "condot_global", "gauss_ot"])

In [ ]:
# §8b: paired tests, weak / small pairs, training validation error
import json, glob
A8 = ["CellOT", "paired", "condot_ct", "condot_global"]
print("paired over the 20 test donor-repeats, vs per-pair unpaired CellOT (same pooling as CondOT)")
display(paired_dr(E8, A8 + ["shift"], "CellOT").round(4))
print("vs the paired control"); display(paired_dr(E8, ["condot_ct", "condot_global"], "paired").round(4))
print("error split: shared part (average response of the 4 test donors) vs patient-specific part, MSE per case")
display(pd.DataFrame({m: {"shared": ((E8["hb_" + m] - E8.rbar) ** 2).mean(), "patient-specific": ((E8["h_" + m] - E8.rc) ** 2).mean()}
                      for m in A8}).T.round(4))
print("skill by stim"); display(skill_by(E8, "stim", A8 + ["shift"]).round(3))
for s in ["IL33", "TNFa", "GMCSF", "PI"]:
    print(f"{s}: paired vs CellOT"); display(paired_dr(E8[E8.stim == s], ["paired", "condot_ct", "condot_global"], "CellOT").round(4))
E8["mean response"] = pd.cut((E8.mu_s - E8.mu_u).abs(), [0, 0.05, 0.1, 0.2, 0.5, np.inf])
print("skill by size of the average response"); display(skill_by(E8, "mean response", A8 + ["shift"]).round(3))
ctn = E8.groupby("celltype").n_u.median().sort_values()
E8["celltype size"] = E8.celltype.map(pd.qcut(ctn, 4, labels=["smallest 7", "2nd", "3rd", "largest 7"]))
print("smallest 7 celltypes:", ", ".join(ctn.index[:7]))
display(pd.concat({"skill": skill_by(E8, "celltype size", A8 + ["shift"]),
                   "specific R²": pd.DataFrame({m: E8.groupby("celltype size").apply(lambda g: spec(g, m)["specific R²"]) for m in A8})}).round(3))
for b, g in E8.groupby("celltype size"):
    print(f"celltype size {b}: vs CellOT"); display(paired_dr(g, ["paired", "condot_ct", "condot_global"], "CellOT").round(4))
PM = E8.groupby(["stim", "celltype"]).apply(lambda g: pd.Series({m: ((g[m] - g.t) ** 2).mean() for m in A8 + ["shift"]}))
PM["CellOT third"] = pd.qcut(PM.CellOT / PM["shift"], 3, labels=["CellOT best vs shift", "middle", "CellOT worst vs shift"])
print("per (stim, celltype) map: share where CondOT has lower error than CellOT, by how CellOT does against the shift")
display(PM.groupby("CellOT third").apply(lambda g: pd.Series({"maps": len(g), "condot_ct better": (g.condot_ct < g.CellOT).mean(),
        "condot_global better": (g.condot_global < g.CellOT).mean(), "median condot_ct / CellOT MSE": (g.condot_ct / g.CellOT).median()})).round(3))
V = []
for f in sorted(glob.glob(str(WORK / "lc/condot/celltype/repeat_*/*/meta.json")) + glob.glob(str(WORK / "lc/condot/global/repeat_*/meta.json"))):
    m = json.load(open(f)); V.append({"arm": "condot_" + ("ct" if m["mode"] == "celltype" else "global"), "best val": m["best_val_median_err"]})
V = pd.DataFrame(V)
cv = []
for d in sorted((WORK / "lc/train").glob("repeat_[0-9]")):
    v = np.load(d / "val_err.npy"); mp = pd.read_csv(d / "maps.csv"); cv += list(np.nanmin(v, 0)[(mp.k == 15).values])
print("validation median error on the TRAINING donors' val cells (lower = better fit; CellOT: best per map, CondOT: one "
      "checkpoint for all its conditions)")
display(pd.concat([V.groupby("arm")["best val"].describe()[["count", "50%", "mean", "max"]],
                   pd.Series(cv).describe()[["count", "50%", "mean", "max"]].to_frame("CellOT k=15 (per map)").T]).round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
ref = dr_mse(E8, "CellOT")
for i, m in enumerate(["paired", "shift", "condot_ct", "condot_global"]):
    r = dr_mse(E8, m) / ref
    ax.scatter(np.full(len(r), i) + np.random.default_rng(0).uniform(-.12, .12, len(r)), r, s=22, color=COL[m], alpha=.8)
    ax.plot([i - .25, i + .25], [r.median()] * 2, c="k", lw=1.5)
    ax.text(i + .28, r.median(), f"{(r < 1).sum()}/20", fontsize=8, va="center")
ax.axhline(1, c="k", lw=.8, ls=":"); ax.set_xticks(range(4)); ax.set_xticklabels(["paired CellOT", "shift", "condot_ct", "condot_global"])
ax.set_ylabel("MSE / CellOT's MSE, per test donor-repeat"); ax.set_yscale("log")
ax.set_yticks([0.6, 0.8, 1, 1.25, 1.5, 2, 2.5]); ax.set_yticklabels(["0.6", "0.8", "1", "1.25", "1.5", "2", "2.5"]); ax.minorticks_off()
ax.set_title("Below 1 = better than per-pair CellOT on that donor (label: donor-repeats below 1)", fontsize=9)
ax.grid(axis="y", alpha=.3); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(FIG / "condot_vs_cellot.png", dpi=160)

**How to read §8.** The first table and the specific-R² tables are laid out as in §7.

`CellOT` is the per-pair unpaired run of the learning curve. It is the like-for-like reference, because CondOT also pools unpaired cells. `paired` is the paired control.

§8b gives:
- paired tests over the 20 test donor-repeats;
- the error split into shared and patient-specific parts;
- skill and paired tests by stim;
- results by response size and celltype size;
- per map, how often CondOT beats CellOT;
- the training-side validation error.

The figure shows one dot per donor-repeat: MSE divided by CellOT's.

On the learning-curve scale the skills are:

| arm | skill |
|---|---|
| CellOT | 0.596 |
| paired | 0.615 |
| shift | 0.604 |
| **condot_ct** | **0.562** |
| **condot_global** | **0.421** |

**Training check.** No run failed or diverged: there was no NaN, and all 140 + 5 tasks completed.
- **condot_ct fits less well than per-pair CellOT even on the training donors.** Its best validation median error is a median 0.042 (range 0.030–0.071; mDCs and B cells are worst). Per-pair CellOT is at 0.021 at k = 15.
  - condot_ct was still improving slowly at 50k iterations: the mean error over successive 10k blocks went 0.065 → 0.059 → 0.056 → 0.055, and the median best step was 40k.
  - Each stim gets about 8k of the 50k iterations, against 50k for each per-pair CellOT map.
- **condot_global is clearly under-fit.** Its best validation error is 0.088–0.096, about 4× CellOT's.
  - It plateaued by about 60k iterations and then oscillated between 0.095 and 0.19. More iterations alone would not fix it.
  - Each of the 168 pairs gets about 900 iterations.
  - There is no identity initialisation: the smoke test started at an error of 143.
  - CondOT's checkpoint is a single one shared by all its conditions, while CellOT picks the best checkpoint for each map.

**Reading (2026-10-07).**
- **Sharing one map across stims does not help.**
  - condot_ct is worse than per-pair CellOT for **19 of 20** donor-repeats (median MSE +10 %, p 6e-6). It is worse than the paired control for 20/20 (+12 %).
  - Skill is 0.621 vs 0.651 (paired 0.666, shift 0.664). Specific R² is 0.037 vs 0.046.
  - condot_global is worse for 20/20 (+77 %), with skill 0.498.
- **The loss is in the average response, not in the donor-specific part.** The patient-specific error is unchanged: 0.0224 for condot_ct and 0.0234 for condot_global, against 0.0222 for CellOT. The shared error is worse: 0.0179 and 0.0418, against 0.0150. So sharing adds no donor-specific information, as expected since the context describes only the stim. It costs accuracy on each stim's mean response.
- **Per stim, condot_ct has lower skill than CellOT in all 7 stims.**
  - GMCSF 0.606 vs 0.632; IFNa 0.686 vs 0.718; IL246 0.601 vs 0.646; LPS 0.604 vs 0.633; PI 0.700 vs 0.715.
  - TNFa is the only stim where its specific R² is higher (0.051 vs 0.025; global 0.072). Its total error there is still worse (better for 5/20).
- **No borrowing of strength for weak or small pairs. They are where sharing hurts most.**
  - IL33: skill 0.038 vs 0.118, better for 2/20 donor-repeats.
  - TNFa: 0.421 vs 0.465, 5/20.
  - Responses under 0.05: 0.117 vs 0.216. 0.05–0.1: 0.442 vs 0.540.
  - The 7 smallest celltypes (pDCs, mDCs, CD56hi NK, CD8Tcm CCR2+, intMCs, ncMCs, CD4Teff): 0.557 vs 0.584, 1/20.
  - Per map, condot_ct beats CellOT on 27 of 168 maps, mostly where CellOT is already worse than the shift (29 % of that third, against 9–11 % elsewhere). There the shift is better still.
- **Answer.** On these data, a CondOT-style map conditioned on the perturbation does not beat per-pair CellOT on held-out donors, overall or on the weak or small pairs. It also cannot reach the donor-specific part, which is the plateau.

**Caveats.**
- The comparison uses CondOT's own budget, not the same number of iterations per condition (about 6× fewer per stim for condot_ct, about 56× fewer per pair for condot_global).
- CondOT uses one checkpoint for all its conditions.
- There are 5 repeats and 14 distinct test donors, so the donor-repeats are not fully independent.

A matched-budget condot_ct (about 300k iterations per celltype) would settle the under-fit question. It would only bring the average response up to CellOT's level, which the shift already provides for free.

## 6. Files and commands

- Case table: `lc/where_skill.py` → `work/lc/where_skill/cases.pkl` (one row per repeat × map × k × test donor ×
  quantile × marker, with the truth, the noise, the training means and SDs, and all predictions).
- This notebook: `lc/build_where_skill_notebook.py` + `lc/where_skill_readings.md` →
  `notebooks/where_cellot_skill.ipynb`; figures in `work/lc/where_skill/`.
- Related: `work/lc/residual_predict` (linear predictors of the residual from Unstim), `work/lc/ctx_score`
  (donor contexts), `notebooks/phenotype_shifts.ipynb` (population sizes), `notebooks/atlas_vs_ool.ipynb` (OOL inputs
  off the atlas range).